# 🌾 CROPIC Crop Analytics System - Interactive Notebook

This notebook provides an interactive environment for:
- Understanding the system components
- Training the model
- Testing predictions
- Analyzing results

## 1. Setup and Imports

In [ ]:
# Install required packages (uncomment if needed)
# !pip install tensorflow opencv-python pillow numpy pandas matplotlib scikit-learn

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import pandas as pd
from pathlib import Path

# Import custom modules
from model_training import CropHealthModel
from image_preprocessor import ImagePreprocessor

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

# Set style for plots
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

## 2. Initialize Model and Preprocessor

In [ ]:
# Initialize model
model = CropHealthModel(num_classes=4, img_size=224)
model.build_model()
model.compile_model()

# Initialize preprocessor
preprocessor = ImagePreprocessor(target_size=(224, 224))

print("Model and preprocessor initialized successfully!")
print(f"\nClass names: {model.class_names}")

## 3. View Model Architecture

In [ ]:
# Display model summary
model.model.summary()

# Count parameters
total_params = model.model.count_params()
print(f"\nTotal Parameters: {total_params:,}")

## 4. Prepare Dataset

Make sure you have organized your dataset in the following structure:
```
crop_dataset/
  train/
    Healthy/
    Pest_Disease/
    Flood_Damage/
    Drought_Stress/
  val/
    Healthy/
    Pest_Disease/
    Flood_Damage/
    Drought_Stress/
```

In [ ]:
# Check dataset structure
train_dir = Path('crop_dataset/train')
val_dir = Path('crop_dataset/val')

if train_dir.exists() and val_dir.exists():
    print("Dataset structure found!\n")
    
    # Count images in each class
    for split in ['train', 'val']:
        print(f"{split.upper()} SET:")
        split_path = Path(f'crop_dataset/{split}')
        
        for class_dir in split_path.iterdir():
            if class_dir.is_dir():
                count = len(list(class_dir.glob('*.[jp][pn][g]')))
                print(f"  {class_dir.name}: {count} images")
        print()
else:
    print("⚠️ Dataset not found. Run prepare_dataset.py first!")

## 5. Visualize Sample Images

In [ ]:
def display_sample_images(dataset_dir, num_samples=2):
    """Display sample images from each class"""
    fig, axes = plt.subplots(4, num_samples, figsize=(12, 16))
    
    for idx, class_name in enumerate(model.class_names):
        class_path = Path(dataset_dir) / class_name
        
        if class_path.exists():
            images = list(class_path.glob('*.[jp][pn][g]'))[:num_samples]
            
            for col, img_path in enumerate(images):
                img = Image.open(img_path)
                axes[idx, col].imshow(img)
                axes[idx, col].axis('off')
                axes[idx, col].set_title(f"{class_name}")
    
    plt.tight_layout()
    plt.show()

# Display samples from training set
if train_dir.exists():
    display_sample_images('crop_dataset/train', num_samples=3)
else:
    print("No training images to display")

## 6. Train the Model

**Note**: Adjust epochs and batch_size based on your dataset size and computational resources.

In [ ]:
# Training parameters
EPOCHS = 10  # Increase for better accuracy
BATCH_SIZE = 32

# Train the model
if train_dir.exists() and val_dir.exists():
    print("Starting training...\n")
    
    history = model.train(
        train_dir='crop_dataset/train',
        val_dir='crop_dataset/val',
        epochs=EPOCHS,
        batch_size=BATCH_SIZE
    )
    
    print("\n✅ Training complete!")
else:
    print("⚠️ Cannot train: Dataset not found")

## 7. Visualize Training History

In [ ]:
def plot_training_history(history):
    """Plot training and validation metrics"""
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Plot accuracy
    axes[0].plot(history.history['accuracy'], label='Train Accuracy')
    axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
    axes[0].set_title('Model Accuracy')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].legend()
    axes[0].grid(True)
    
    # Plot loss
    axes[1].plot(history.history['loss'], label='Train Loss')
    axes[1].plot(history.history['val_loss'], label='Val Loss')
    axes[1].set_title('Model Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True)
    
    plt.tight_layout()
    plt.show()
    
    # Print final metrics
    print("\nFinal Metrics:")
    print(f"Train Accuracy: {history.history['accuracy'][-1]:.4f}")
    print(f"Val Accuracy: {history.history['val_accuracy'][-1]:.4f}")
    print(f"Train Loss: {history.history['loss'][-1]:.4f}")
    print(f"Val Loss: {history.history['val_loss'][-1]:.4f}")

# Plot if training was performed
if 'history' in locals():
    plot_training_history(history)
else:
    print("No training history to display")

## 8. Save the Trained Model

In [ ]:
# Save the model
if 'history' in locals():
    model.save_model('crop_health_model.h5')
    print("✅ Model saved successfully!")
    print("\nModel files:")
    print("  - crop_health_model.h5")
    print("  - class_names.json")
else:
    print("⚠️ No trained model to save")

## 9. Load Model and Test Predictions

In [ ]:
# Load the saved model
test_model = CropHealthModel()

try:
    test_model.load_model('crop_health_model.h5')
    print("✅ Model loaded successfully!")
    print(f"Classes: {test_model.class_names}")
except:
    print("⚠️ No saved model found. Using untrained model for demo.")
    test_model = model

## 10. Test with Single Image

In [ ]:
def test_single_image(image_path, model, preprocessor):
    """Test prediction on a single image"""
    # Load and display original image
    img = Image.open(image_path)
    
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Original image
    axes[0].imshow(img)
    axes[0].set_title('Original Image')
    axes[0].axis('off')
    
    # Validate quality
    validation = preprocessor.validate_image_quality(img)
    
    # Preprocess and predict
    processed_img = preprocessor.preprocess_image(img)
    prediction = model.predict(processed_img)
    
    # Display results
    axes[1].barh(range(len(prediction['all_probabilities'])),
                 [v for v in prediction['all_probabilities'].values()])
    axes[1].set_yticks(range(len(prediction['all_probabilities'])))
    axes[1].set_yticklabels(prediction['all_probabilities'].keys())
    axes[1].set_xlabel('Confidence')
    axes[1].set_title('Prediction Probabilities')
    axes[1].set_xlim([0, 1])
    
    plt.tight_layout()
    plt.show()
    
    # Print results
    print("\n" + "="*50)
    print("PREDICTION RESULTS")
    print("="*50)
    print(f"\nPredicted Class: {prediction['predicted_class']}")
    print(f"Confidence: {prediction['confidence']:.2%}")
    print(f"\nQuality Score: {validation['metrics']['quality_score']:.1f}/100")
    print(f"\nTop 3 Predictions:")
    for i, pred in enumerate(prediction['top_3_predictions'], 1):
        print(f"  {i}. {pred['class']}: {pred['confidence']:.2%}")
    
    if validation['issues']:
        print(f"\n⚠️ Quality Issues:")
        for issue in validation['issues']:
            print(f"  - {issue}")

# Test with an image (replace with your image path)
# test_single_image('path/to/your/test/image.jpg', test_model, preprocessor)

## 11. Batch Testing and Evaluation

In [ ]:
def evaluate_on_validation_set(val_dir, model, preprocessor):
    """Evaluate model on validation set"""
    results = []
    
    for class_name in model.class_names:
        class_path = Path(val_dir) / class_name
        
        if not class_path.exists():
            continue
        
        images = list(class_path.glob('*.[jp][pn][g]'))[:10]  # Test first 10
        
        for img_path in images:
            try:
                img = Image.open(img_path)
                processed = preprocessor.preprocess_image(img)
                prediction = model.predict(processed)
                
                results.append({
                    'true_class': class_name,
                    'predicted_class': prediction['predicted_class'],
                    'confidence': prediction['confidence'],
                    'correct': class_name == prediction['predicted_class']
                })
            except Exception as e:
                print(f"Error processing {img_path}: {e}")
    
    # Create results dataframe
    df = pd.DataFrame(results)
    
    # Calculate metrics
    accuracy = df['correct'].mean()
    
    print(f"\n{'='*50}")
    print(f"VALIDATION RESULTS")
    print(f"{'='*50}")
    print(f"\nTotal Images: {len(df)}")
    print(f"Accuracy: {accuracy:.2%}")
    print(f"\nPer-Class Accuracy:")
    
    for class_name in model.class_names:
        class_df = df[df['true_class'] == class_name]
        if len(class_df) > 0:
            class_acc = class_df['correct'].mean()
            print(f"  {class_name}: {class_acc:.2%} ({len(class_df)} images)")
    
    return df

# Evaluate if validation set exists
if val_dir.exists():
    results_df = evaluate_on_validation_set('crop_dataset/val', test_model, preprocessor)
else:
    print("No validation set found")

## 12. Create Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

def plot_confusion_matrix(results_df, class_names):
    """Plot confusion matrix"""
    y_true = results_df['true_class']
    y_pred = results_df['predicted_class']
    
    cm = confusion_matrix(y_true, y_pred, labels=class_names)
    
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names,
                yticklabels=class_names)
    plt.title('Confusion Matrix')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.show()

if 'results_df' in locals() and len(results_df) > 0:
    plot_confusion_matrix(results_df, test_model.class_names)
else:
    print("No results to plot confusion matrix")

## 13. Summary and Next Steps

In [ ]:
print("\n" + "="*60)
print("NOTEBOOK SUMMARY")
print("="*60)

print("\n✅ Completed Steps:")
print("  1. Initialized model and preprocessor")
print("  2. Viewed model architecture")
if train_dir.exists():
    print("  3. Checked dataset structure")
if 'history' in locals():
    print("  4. Trained the model")
    print("  5. Visualized training history")
    print("  6. Saved the model")
if 'results_df' in locals():
    print("  7. Evaluated on validation set")
    print("  8. Created confusion matrix")

print("\n📝 Next Steps:")
print("  1. Run the web application: streamlit run app.py")
print("  2. Test with real crop images")
print("  3. Fine-tune the model for better accuracy")
print("  4. Deploy to production environment")

print("\n" + "="*60)